In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotnine as p9
from liesel_ptm.bspline.onion import OnionKnots, OnionSpline

from ppptm.nodes.ppvar_rw import GPTMCoef
from ppptm.util.locs import unit_grid_vars

jax.config.update("jax_enable_x64", True)

In [ ]:
D = 10
knots = OnionKnots(-4.0, 4.0, nparam=D).knots
bs = OnionSpline(knots)

In [ ]:
locs = unit_grid_vars()
coef = GPTMCoef.sample_array(
    locs=locs,
    nparam=D,
    n=1,
    length_scale=0.1,
    amplitude=0.05,
)
# coef = jnp.swapaxes(coef, 0, 1)
print(coef.shape)

In [ ]:
xgrid = jnp.linspace(-8.0, 8.0, 300)
x = jnp.expand_dims(xgrid, -1)
x = jnp.tile(x, (1, locs.locs.nloc))
x.shape

In [ ]:
fx, fx_deriv = bs.dot_and_deriv(x, coef.squeeze())
print(fx.shape)
print(fx_deriv.shape)

In [ ]:
locsdf = pd.DataFrame(locs.locs.ordered, columns=["lon", "lat"]).reset_index(
    names="loc"
)

In [ ]:
nobs, nloc = fx.shape

fx_long = jnp.reshape(fx, (nobs, nloc))
df = pd.DataFrame(fx_long)
df["x"] = xgrid
df["nobs"] = np.arange(x.shape[0])
df = df.melt(id_vars=["x", "nobs"], var_name="loc")
# df["sample"] = np.repeat(np.arange(nsamp), )
df = pd.merge(df, locsdf, on="loc")
df["loc"] = df["loc"].astype(int)
df

In [ ]:
(p9.ggplot(df) + p9.geom_line(p9.aes("x", "value", group="loc", color="lon")))

In [ ]:
(p9.ggplot(df) + p9.geom_line(p9.aes("x", "value", group="loc", color="loc")))

In [ ]:
(p9.ggplot(df) + p9.geom_line(p9.aes("x", "value", group="loc", color="lat")))